# RAG Pipeline OP — Cluster-Aware Retrieval
This notebook is an upgraded version of `rag-pipeline.ipynb`.
The core improvement is **cluster-aware retrieval** — instead of searching all 207 chunks,
the pipeline first predicts which cluster the question belongs to, then searches only
that cluster's chunks. This produces a smaller, more focused candidate set before FAISS scoring.

| Step | What happens |
|---|---|
| **Step 0** | Setup — libraries, config, load all artifacts |
| **Step 1** | Load Embedding Model + FAISS Index + Clustered Chunks |
| **Step 2** | Cluster Prediction — assign incoming question to a cluster |
| **Step 3** | Cluster-Aware Retrieval — FAISS search scoped to predicted cluster |
| **Step 4** | Relevance Threshold Check |
| **Step 5** | Prompt Construction |
| **Step 6** | LLM Call (Groq) |
| **Step 7** | Full Pipeline Function (`rag_answer_v2`) |
| **Step 8** | Interactive Mode — ask your own question |
| **Step 9** | Batch Testing — same 8 test questions as v1 |
| **Step 10** | Evaluation — Hit Rate + MRR |
| **Step 11** | v1 vs v2 Comparison Table |
| **Step 12** | Results Table + Save to JSON |

---

> **Before running this notebook, make sure:**
> 1. `rag-pipeline.ipynb` has been fully run (v1 baseline)
> 2. `clustering.ipynb` has been fully run
> 3. `data/clustering/chunks_with_clusters.json` exists
> 4. `data/embeddings/faiss_index.bin` exists
> 5. Your `.env` file has `GROQ_API_KEY=your_key_here` in the project root

---

**What changes from v1 to v2:**

```
v1 — Original RAG:
   Question → Embed → FAISS search (all 207 chunks) → Top-3 → LLM

v2 — Cluster-Aware RAG:
   Question → Embed → Predict cluster → FAISS search (only that cluster's chunks)
                                                        → Top-3 → LLM
```

In [2]:
import os
import json
import numpy as np
import pandas as pd
import faiss
from groq import Groq
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

print("Imports successful")

Imports successful


In [3]:
# Load API key from .env file
load_dotenv("../.env")

# Directory paths
EMBEDDINGS_DIR  = Path("../../data/embeddings")
CLUSTERING_DIR  = Path("../../data/clustering")
RAG_DIR         = Path("../../data/rag")

# Model config
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL       = "llama-3.1-8b-instant"   # Groq-hosted Llama 3.1 8B — current production model
MAX_TOKENS      = 300

# Retrieval config
TOP_K           = 3      # Number of chunks to retrieve per question
SIM_THRESHOLD   = 0.3    # Min similarity score — below this = no answer

# Cluster-aware retrieval config
# When the predicted cluster has fewer than MIN_CLUSTER_CHUNKS chunks,
# fall back to full-index search to avoid missing relevant results.
MIN_CLUSTER_CHUNKS = TOP_K   # must have at least TOP_K chunks to search within

# Groq client — reads GROQ_API_KEY from environment
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

RAG_DIR.mkdir(parents=True, exist_ok=True)

# Verify setup
groq_key = os.getenv("GROQ_API_KEY")
print(f"Groq API key        : {'Loaded' if groq_key else 'NOT FOUND — check your .env file'}")
print(f"Embedding model     : {EMBED_MODEL}")
print(f"LLM model           : {LLM_MODEL}")
print(f"Top-K retrieval     : {TOP_K}")
print(f"Similarity threshold: {SIM_THRESHOLD}")
print(f"Min cluster chunks  : {MIN_CLUSTER_CHUNKS}")

Groq API key        : Loaded
Embedding model     : sentence-transformers/all-MiniLM-L6-v2
LLM model           : llama-3.1-8b-instant
Top-K retrieval     : 3
Similarity threshold: 0.3
Min cluster chunks  : 3


---
## Load Embedding Model + FAISS Index + Clustered Chunks

We load four things in this step:

- **Embedding model** — same `all-MiniLM-L6-v2` used across the entire pipeline
- **FAISS index** — the full vector store built in `04_embedding.ipynb`
- **Clustered chunks** — `chunks_with_clusters.json` from `05_clustering.ipynb`; same as `chunks_with_metadata.json` but with `cluster_id` and `cluster_label` added to every chunk
- **Cluster centroids** — the K-Means centroid vectors extracted from the fitted model; used to predict which cluster a new question belongs to

The cluster centroids are reconstructed from the cluster assignments by averaging the embeddings
of all chunks in each cluster — this avoids needing to save the KMeans model separately.

In [4]:
# Load embedding model
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")
embed_model.max_seq_length = 128   # FAQ chunks are short — cap saves compute on every encode() call
print(f"Embedding model loaded | Dimensions: {embed_model.get_sentence_embedding_dimension()}")

# Load FAISS index
faiss_path = EMBEDDINGS_DIR / "faiss_index.bin"
index = faiss.read_index(str(faiss_path))
print(f"\nFAISS index loaded")
print(f"   Vectors in index : {index.ntotal}")
print(f"   Dimensions       : {index.d}")

# Load clustered chunk metadata (output of 05_clustering.ipynb)
clusters_path = CLUSTERING_DIR / "chunks_with_clusters.json"
with open(clusters_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"\nClustered chunks loaded")
print(f"   Total chunks     : {len(chunks)}")
print(f"   Fields per chunk : {list(chunks[0].keys())}")

# Sanity check — FAISS index and chunk list must be in sync
assert index.ntotal == len(chunks), (
    f"Mismatch: FAISS has {index.ntotal} vectors but chunk list has {len(chunks)} entries. "
    "Re-run 04_embedding.ipynb and 05_clustering.ipynb."
)
print("\nFAISS index and chunk metadata are in sync")

Loading embedding model...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2944.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded | Dimensions: 384

FAISS index loaded
   Vectors in index : 207
   Dimensions       : 384

Clustered chunks loaded
   Total chunks     : 207
   Fields per chunk : ['chunk_id', 'bank_name', 'source_file', 'page_number', 'chunk_index', 'token_count', 'text', 'cluster_id', 'cluster_label']

FAISS index and chunk metadata are in sync


In [5]:
# Build cluster index structures needed for cluster-aware retrieval
#
# cluster_to_indices : dict mapping cluster_id -> list of FAISS index positions
#                      Used to know which chunk positions belong to each cluster
# cluster_centroids  : dict mapping cluster_id -> mean embedding vector of that cluster
#                      Used to predict cluster for a new question via nearest centroid

print("Building cluster index structures...")

# Determine number of clusters from the data
all_cluster_ids = [c["cluster_id"] for c in chunks]
K_FINAL         = max(all_cluster_ids) + 1

# Map each cluster_id to the list of chunk positions (FAISS indices) in that cluster
cluster_to_indices = {k: [] for k in range(K_FINAL)}
for position, chunk in enumerate(chunks):
    cluster_to_indices[chunk["cluster_id"]].append(position)

# Reconstruct cluster centroids by averaging the raw embeddings of all chunks in each cluster
# We use the FAISS index to retrieve the stored vectors (reconstruct method)
cluster_centroids = {}
for k in range(K_FINAL):
    positions = cluster_to_indices[k]
    # Retrieve stored vectors for all positions in this cluster
    cluster_vecs = np.array([
        index.reconstruct(int(pos)) for pos in positions
    ], dtype=np.float32)
    # Mean vector = centroid; re-normalize to unit length
    centroid = cluster_vecs.mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    cluster_centroids[k] = centroid

print(f"Clusters found     : {K_FINAL}")
print(f"\nCluster sizes:")
for k in range(K_FINAL):
    label = chunks[cluster_to_indices[k][0]]["cluster_label"]
    print(f"   Cluster {k} ({len(cluster_to_indices[k]):3d} chunks) : {label}")

print("\nCluster index structures ready")

Building cluster index structures...
Clusters found     : 6

Cluster sizes:
   Cluster 0 ( 39 chunks) : Digital Accounts
   Cluster 1 ( 36 chunks) : Islamic Banking
   Cluster 2 ( 36 chunks) : Loans & Financing Products
   Cluster 3 ( 33 chunks) : Online Services
   Cluster 4 ( 26 chunks) : International Transfers & Remittances
   Cluster 5 ( 37 chunks) : State Bank of Pakistan & Regulations

Cluster index structures ready


---
## Cluster Prediction

Before searching FAISS, we predict which cluster the incoming question belongs to.

**How it works:**
1. Embed the question using the same `all-MiniLM-L6-v2` model
2. Compute cosine similarity between the question vector and each cluster centroid
3. The cluster with the highest similarity score is the predicted cluster
4. FAISS search is then scoped to only the chunks in that cluster

**Why centroid-based prediction:**
- Simple and fast — just K dot products, no model inference needed
- Consistent with K-Means geometry — the centroid is the mean of all chunk vectors in a cluster
- No extra model to train or save — centroids are derived directly from the clustering output

**Fallback:**
If the predicted cluster has fewer than `MIN_CLUSTER_CHUNKS` chunks (set to `TOP_K = 3`),
the pipeline falls back to full-index search across all chunks. This protects against
very small clusters where we could not retrieve enough distinct results.

In [6]:
def predict_cluster(question_vec: np.ndarray) -> dict:
    """
    Predict the most relevant cluster for a question vector
    by computing cosine similarity against all cluster centroids.

    Args:
        question_vec : L2-normalized question embedding, shape (1, 384)

    Returns:
        dict with keys:
        - predicted_cluster_id    (int)
        - predicted_cluster_label (str)
        - similarity_scores       (dict: cluster_id -> score)
        - fallback                (bool) True if cluster is too small
    """
    q_vec = question_vec[0]  # shape (384,)

    # Compute cosine similarity between question and each centroid
    # Both are L2-normalized so dot product = cosine similarity
    scores = {
        k: float(np.dot(q_vec, cluster_centroids[k]))
        for k in range(K_FINAL)
    }

    # Predicted cluster = highest similarity
    predicted_id    = max(scores, key=scores.get)
    predicted_label = chunks[cluster_to_indices[predicted_id][0]]["cluster_label"]

    # Check if predicted cluster has enough chunks to retrieve TOP_K results
    cluster_size = len(cluster_to_indices[predicted_id])
    fallback     = cluster_size < MIN_CLUSTER_CHUNKS

    return {
        "predicted_cluster_id":    predicted_id,
        "predicted_cluster_label": predicted_label,
        "similarity_scores":       scores,
        "cluster_size":            cluster_size,
        "fallback":                fallback
    }


# Quick test — predict cluster for two questions
print("Testing cluster prediction...\n")

test_questions = [
    "What documents are needed to open a Meezan Roshan Digital Account?",
    "What is the late payment charge on Alfalah personal loan?"
]

for q in test_questions:
    q_vec = embed_model.encode(
        [q], normalize_embeddings=True,
        batch_size=1, show_progress_bar=False, convert_to_numpy=True
    ).astype(np.float32)

    pred = predict_cluster(q_vec)
    print(f"Question : {q[:65]}")
    print(f"   Predicted cluster : {pred['predicted_cluster_id']} — {pred['predicted_cluster_label']}")
    print(f"   Cluster size      : {pred['cluster_size']} chunks")
    print(f"   Fallback          : {pred['fallback']}")
    sorted_scores = sorted(pred['similarity_scores'].items(), key=lambda x: x[1], reverse=True)
    print(f"   All cluster scores: {[(k, round(v, 4)) for k, v in sorted_scores]}")
    print()

Testing cluster prediction...



Question : What documents are needed to open a Meezan Roshan Digital Account
   Predicted cluster : 0 — Digital Accounts
   Cluster size      : 39 chunks
   Fallback          : False
   All cluster scores: [(0, 0.676), (4, 0.435), (2, 0.4231), (5, 0.4228), (3, 0.4143), (1, 0.3787)]

Question : What is the late payment charge on Alfalah personal loan?
   Predicted cluster : 4 — International Transfers & Remittances
   Cluster size      : 26 chunks
   Fallback          : False
   All cluster scores: [(4, 0.4821), (2, 0.4572), (3, 0.2988), (1, 0.2955), (0, 0.2911), (5, 0.2146)]



---
## Cluster-Aware Retrieval

This is the core upgrade from v1. Instead of searching all chunks in FAISS, we:

1. Predict the cluster for the question
2. Extract the FAISS positions of only the chunks in that cluster
3. Retrieve those vectors from the FAISS index
4. Score them manually against the question vector using dot product (cosine similarity)
5. Return the Top-K highest scoring chunks

**Why not use a separate per-cluster FAISS index:**
Building K separate FAISS indexes would require more storage and setup. Instead, we use
`index.reconstruct()` to pull stored vectors for specific positions, then score them
ourselves. For clusters of 20-50 chunks this is equally fast and avoids managing multiple index files.

**Fallback behavior:**
If the predicted cluster is too small (`< MIN_CLUSTER_CHUNKS`), the function automatically
falls back to a full FAISS search across all chunks — same behavior as v1.

In [7]:
def retrieve_chunks_v2(question: str, top_k: int = TOP_K) -> tuple[list[dict], dict]:
    """
    Cluster-aware retrieval: predict question cluster, then search only
    that cluster's chunks instead of the full FAISS index.

    Args:
        question : User's question string.
        top_k    : Number of chunks to return.

    Returns:
        Tuple of:
        - List of retrieved chunk dicts with rank and similarity added
        - Cluster prediction info dict
    """
    # Step 1 — Embed the question
    question_vec = embed_model.encode(
        [question],
        normalize_embeddings=True,
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32)

    # Step 2 — Predict cluster
    cluster_pred = predict_cluster(question_vec)

    if cluster_pred["fallback"]:
        # Fallback: cluster too small — search full index (same as v1)
        scores, indices = index.search(question_vec, top_k)
        result_positions = list(indices[0])
        result_scores    = list(scores[0])
    else:
        # Cluster-aware: retrieve vectors for only this cluster's chunk positions
        candidate_positions = cluster_to_indices[cluster_pred["predicted_cluster_id"]]

        # Reconstruct stored vectors for each candidate position
        candidate_vecs = np.array([
            index.reconstruct(int(pos)) for pos in candidate_positions
        ], dtype=np.float32)  # shape: (cluster_size, 384)

        # Score all candidate vectors against the question vector
        # dot product on L2-normalized vectors = cosine similarity
        raw_scores = candidate_vecs.dot(question_vec[0])  # shape: (cluster_size,)

        # Sort by descending score and take top_k
        top_local_indices = np.argsort(raw_scores)[::-1][:top_k]
        result_positions  = [candidate_positions[i] for i in top_local_indices]
        result_scores     = [float(raw_scores[i]) for i in top_local_indices]

    # Step 3 — Build result list with metadata
    retrieved = []
    for rank, (pos, score) in enumerate(zip(result_positions, result_scores), start=1):
        chunk = chunks[pos].copy()
        chunk["rank"]       = rank
        chunk["similarity"] = round(float(score), 4)
        retrieved.append(chunk)

    return retrieved, cluster_pred


# Quick test
print("Testing cluster-aware retrieval...\n")
test_retrieved, test_pred = retrieve_chunks_v2("How to open a bank account in Pakistan?")

print(f"Predicted cluster : {test_pred['predicted_cluster_id']} — {test_pred['predicted_cluster_label']}")
print(f"Searched          : {test_pred['cluster_size']} chunks (instead of {len(chunks)})")
print(f"Fallback used     : {test_pred['fallback']}")
print()
for r in test_retrieved:
    print(f"Rank {r['rank']} | Similarity: {r['similarity']} | {r['bank_name']}")
    print(f"  Source : {r['source_file']} | Page {r['page_number']}")
    print(f"  Text   : {r['text'][:200]}...")
    print()

Testing cluster-aware retrieval...

Predicted cluster : 0 — Digital Accounts
Searched          : 39 chunks (instead of 207)
Fallback used     : False

Rank 1 | Similarity: 0.6935 | Meezan Bank
  Source : Meezan-Bank-FAQs-Roshan-Digital-Account.pdf | Page 2
  Text   : an Digital Account.
18. How do I apply for Internet Banking facility for my Meezan Roshan Digital account?
Please refer to detailed guidelines for Internet banking facilities available at https://www....

Rank 2 | Similarity: 0.6862 | Habib Bank Limited (HBL)
  Source : HBL-Work-Conventional-Accounts.pdf | Page 3
  Text   : Requirements to open an account: To open the account you will 
need to satisfy some identiﬁcation requirements as per regulatory 
instructions and banks' internal policies. These may include providing...

Rank 3 | Similarity: 0.6575 | Meezan Bank
  Source : Meezan-Bank-FAQs-Roshan-Digital-Account.pdf | Page 1
  Text   : FAQs of Meezan Roshan Digital Account 
1.
What is Meezan Roshan Digital Account?
Mee

---
## Relevance Threshold Check

Same logic as v1 — if the best similarity score across retrieved chunks is below `SIM_THRESHOLD`,
the question is considered out-of-scope and the LLM call is skipped entirely.

**Why keep this check in v2:**
Even with cluster-aware retrieval, an out-of-scope question will still be assigned to
the closest cluster with a low similarity score. The threshold check catches this and
returns the fallback answer instead of sending irrelevant chunks to the LLM.

In [8]:
def check_relevance(retrieved: list[dict]) -> dict:
    """
    Check if retrieved chunks pass the similarity threshold.

    Returns:
        dict with keys:
        - is_relevant     (bool)
        - best_similarity (float)
        - status          (str): 'relevant' | 'irrelevant' | 'no_results'
    """
    if not retrieved:
        return {"is_relevant": False, "best_similarity": 0.0, "status": "no_results"}

    best_similarity = max(c["similarity"] for c in retrieved)
    is_relevant     = best_similarity >= SIM_THRESHOLD

    return {
        "is_relevant":     is_relevant,
        "best_similarity": best_similarity,
        "status":          "relevant" if is_relevant else "irrelevant"
    }


# Test both cases
in_chunks,  in_pred  = retrieve_chunks_v2("What is the cash withdrawal limit at Meezan Bank?")
out_chunks, out_pred = retrieve_chunks_v2("What is the capital of France?")

check_in  = check_relevance(in_chunks)
check_out = check_relevance(out_chunks)

print(f"In-scope query  | Best sim: {check_in['best_similarity']:<6} | Status: {check_in['status']}")
print(f"Out-scope query | Best sim: {check_out['best_similarity']:<6} | Status: {check_out['status']}")

In-scope query  | Best sim: 0.6465 | Status: relevant
Out-scope query | Best sim: 0.1304 | Status: irrelevant


---
## Prompt Construction

Identical to v1 — the prompt structure, system instructions, and context format are unchanged.
The only difference is that the retrieved chunks now come from a smaller, more focused
cluster-scoped search rather than the full index.

In [9]:
def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    """
    Build a grounded RAG prompt using retrieved chunks as context.
    The LLM is instructed to answer ONLY from this context.

    Args:
        question         : User's question string.
        retrieved_chunks : List of retrieved chunk dicts from retrieve_chunks_v2().

    Returns:
        Complete prompt string ready to send to the LLM.
    """
    # Format each chunk with a labelled source header
    context_parts = []
    for chunk in retrieved_chunks:
        header = (
            f"[Source {chunk['rank']}: {chunk['bank_name']} "
            f"| {chunk['source_file']} | Page {chunk['page_number']}]"
        )
        context_parts.append(f"{header}\n{chunk['text']}")

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant for Pakistani banking customers.
Answer the question ONLY based on the context documents provided below.
If the answer is not in the context, say exactly: "I don't have enough information to answer this from the available documents."
Do NOT make up any information. Be concise and accurate.
If amounts, limits, or numbers are mentioned in the context, always include them in your answer.

=== CONTEXT DOCUMENTS ===
{context}
=== END OF CONTEXT ===

QUESTION: {question}

ANSWER"""

    return prompt


# Preview sample prompt
sample_question          = "What is the late payment fee for Alfalah personal loan?"
sample_chunks, sample_pred = retrieve_chunks_v2(sample_question)
sample_prompt            = build_prompt(sample_question, sample_chunks)

print("Sample prompt preview (first 1000 chars):")
print("=" * 60)
print(sample_prompt[:1000])
print("...")

Sample prompt preview (first 1000 chars):
You are a helpful assistant for Pakistani banking customers.
Answer the question ONLY based on the context documents provided below.
If the answer is not in the context, say exactly: "I don't have enough information to answer this from the available documents."
Do NOT make up any information. Be concise and accurate.
If amounts, limits, or numbers are mentioned in the context, always include them in your answer.

=== CONTEXT DOCUMENTS ===
[Source 1: Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf | Page 3]
FAQs – Personal Loan 

17. How will I receive my loan? 
 
If you filled and signed IBFT form then approved loan amount will be transferred directly in 
your account via IBFT. Otherwise, a Pay Order against your Alfalah Personal Loan will be 
issued which you can deposit in any of your existing accounts. 
 
18. How long will it take for my case to be processed? 
 
Alfalah Personal Loan is processed within 11 working days of application, sub

---
## LLM Call — Llama 3.1 8B (Groq)

Identical to v1 — the Groq API call, model, and generation parameters are unchanged.
The improvement in response quality (if any) comes from the more focused context
retrieved by the cluster-aware step, not from changes to the LLM call itself.

In [10]:
def call_llm(prompt: str) -> str:
    """
    Send the RAG prompt to Llama 3.1 8B via Groq API and return the answer.
    Uses the Groq Python SDK — requires GROQ_API_KEY in .env.

    Args:
        prompt : The complete RAG prompt string.

    Returns:
        Model answer as a plain string.
    """
    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=MAX_TOKENS,
        temperature=0.1,   # low temperature = factual, grounded answers
        top_p=0.9,
    )
    return response.choices[0].message.content.strip()


# Connectivity test
print("Testing LLM connection...")
test_response = call_llm("Say exactly the words: LLM connection successful")
print(f"LLM Response: {test_response}")

Testing LLM connection...
LLM Response: LLM connection successful.


---
## Full RAG Pipeline v2 (`rag_answer_v2`)

All steps combined into one function:

```
question
    → embed question
    → predict cluster (centroid similarity)
    → retrieve Top-K from that cluster only
    → similarity threshold check
    → build prompt
    → call Groq LLM
    → return structured result
```

The result dict includes a `cluster_info` field not present in v1, which records
which cluster was used and whether fallback was triggered.

In [11]:
# Fallback answer returned when no relevant chunks are found
FALLBACK_ANSWER = "No relevant information found in the available banking documents."


def rag_answer_v2(question: str, verbose: bool = True) -> dict:
    """
    Complete Cluster-Aware RAG Pipeline v2:
      1. Embed the question
      2. Predict cluster via centroid similarity
      3. Retrieve Top-K chunks from predicted cluster
      4. Check similarity threshold
      5. Build grounded prompt
      6. Call Groq LLM
      7. Return structured result

    Args:
        question (str)  : User's question.
        verbose  (bool) : Print step-by-step output.

    Returns:
        dict with keys: question, answer, sources, best_similarity,
                        status, cluster_info, _retrieved
    """
    if verbose:
        print(f"\n{'='*65}")
        print(f"QUESTION: {question}")
        print(f"{'='*65}")

    # Cluster-aware retrieval
    retrieved, cluster_pred = retrieve_chunks_v2(question, top_k=TOP_K)

    if verbose:
        fallback_note = " [FALLBACK: full index]" if cluster_pred["fallback"] else ""
        print(f"\nCluster predicted : {cluster_pred['predicted_cluster_id']} — "
              f"{cluster_pred['predicted_cluster_label']}{fallback_note}")
        print(f"Search space      : {cluster_pred['cluster_size']} chunks "
              f"(vs {len(chunks)} in v1)")
        print(f"\nRetrieved {len(retrieved)} chunks:")
        for c in retrieved:
            print(f"   [{c['similarity']}] {c['bank_name']} | {c['source_file']}")

    # Relevance threshold check
    relevance       = check_relevance(retrieved)
    best_similarity = relevance["best_similarity"]

    if not relevance["is_relevant"]:
        if verbose:
            print(f"\nBest similarity ({best_similarity}) is below threshold ({SIM_THRESHOLD})")
            print("Not enough relevant context found — skipping LLM call.")
        return {
            "question":        question,
            "answer":          FALLBACK_ANSWER,
            "sources":         [],
            "best_similarity": best_similarity,
            "status":          "below_threshold",
            "cluster_info":    cluster_pred,
            "_retrieved":      retrieved,
        }

    # Build prompt
    prompt = build_prompt(question, retrieved)

    # Call LLM
    if verbose:
        print(f"\nSending to Groq ({LLM_MODEL})...")

    answer  = call_llm(prompt)
    sources = list(set(c["source_file"] for c in retrieved))

    if verbose:
        print(f"\nANSWER:")
        print(f"   {answer}")
        print(f"\nSOURCES USED:")
        for s in sources:
            print(f"   - {s}")

    return {
        "question":        question,
        "answer":          answer,
        "sources":         sources,
        "best_similarity": best_similarity,
        "status":          "answered",
        "cluster_info":    cluster_pred,
        # Store retrieved chunks so evaluation step can reuse them without re-embedding
        "_retrieved":      retrieved,
    }


print("rag_answer_v2() function defined and ready.")

rag_answer_v2() function defined and ready.


---
## Interactive Mode

Change the question below and run the cell to test any question against the v2 pipeline.

In [12]:
# Change this question and run the cell
MY_QUESTION = "What is the minimum age requirement for Alfalah personal loan?"

result = rag_answer_v2(MY_QUESTION, verbose=True)


QUESTION: What is the minimum age requirement for Alfalah personal loan?

Cluster predicted : 2 — Loans & Financing Products
Search space      : 36 chunks (vs 207 in v1)

Retrieved 3 chunks:
   [0.6798] Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf
   [0.5834] Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf
   [0.5528] Meezan Bank | Meezan-Bank-FAQs-Salaried.pdf

Sending to Groq (llama-3.1-8b-instant)...

ANSWER:
   The minimum age requirement for Alfalah personal loan is 21 years for salaried individuals and 65 years for SEB/SEP individuals.

SOURCES USED:
   - Bank-Alfalah-FAQs-Personal-Loan.pdf
   - Meezan-Bank-FAQs-Salaried.pdf


---
## Batch Testing

Running the **same 8 test questions** used in `rag-pipeline.ipynb` so that
results are directly comparable between v1 and v2 in Step 11.

In [13]:
# Same 8 test questions as v1 — kept identical for fair comparison
# Format: { "question": str, "expected_source": str | None }

TEST_QUESTIONS = [
    # HBL — Home Remittance
    {"question": "What is the cash transaction limit for HBL home remittance service?",
     "expected_source": "HBL-FAQs-Home-Remittance"},

    # Meezan — Roshan Digital Account
    {"question": "What documents are needed to open a Meezan Roshan Digital Account?",
     "expected_source": "Meezan-Bank-FAQs-Roshan-Digital-Account"},

    # Meezan — Roshan Apna Ghar (solar financing)
    {"question": "Is net metering included in Meezan Bank solar panel financing?",
     "expected_source": "Meezan-Bank-FAQs-Roshan-Apna-Ghar"},

    # Bank Alfalah — Personal Loan
    {"question": "What is the late payment charge on Alfalah personal loan?",
     "expected_source": "Bank-Alfalah-FAQs-Personal-Loan"},

    # ABL — Digital Banking
    {"question": "Can a foreign national register for ABL myABL digital banking?",
     "expected_source": "ABL"},

    # State Bank of Pakistan
    {"question": "What is the withholding tax rate on Pakistan Investment Bonds?",
     "expected_source": "State-Bank"},

    # HBL Islamic
    {"question": "Can Islamic banks charge penalty for late payment?",
     "expected_source": "HBL-Islamic"},

    # Out-of-scope — must trigger below_threshold
    {"question": "What is the capital of France?",
     "expected_source": None},
]

print(f"{len(TEST_QUESTIONS)} test questions loaded")
print(f"   {len([q for q in TEST_QUESTIONS if q['expected_source']])} in-scope questions")
print(f"   {len([q for q in TEST_QUESTIONS if not q['expected_source']])} out-of-scope question")

8 test questions loaded
   7 in-scope questions
   1 out-of-scope question


In [14]:
# Run all test questions through rag_answer_v2 and collect results
all_results = []

for question_item in TEST_QUESTIONS:
    result = rag_answer_v2(question_item["question"], verbose=True)
    result["expected_source"] = question_item["expected_source"]
    all_results.append(result)
    print()


QUESTION: What is the cash transaction limit for HBL home remittance service?

Cluster predicted : 4 — International Transfers & Remittances
Search space      : 26 chunks (vs 207 in v1)

Retrieved 3 chunks:
   [0.7264] Habib Bank Limited (HBL) | HBL-FAQs-Home-Remittance.pdf
   [0.5842] Habib Bank Limited (HBL) | HBL-Work-Conventional-Accounts.pdf
   [0.4813] Habib Bank Limited (HBL) | HBL-Work-Conventional-Accounts.pdf

Sending to Groq (llama-3.1-8b-instant)...



ANSWER:
   The cash transaction limit for HBL home remittance service is PKR 500,000/- per transaction via HBL's Cash Over the Counter (CoC) service.

SOURCES USED:
   - HBL-Work-Conventional-Accounts.pdf
   - HBL-FAQs-Home-Remittance.pdf


QUESTION: What documents are needed to open a Meezan Roshan Digital Account?

Cluster predicted : 0 — Digital Accounts
Search space      : 39 chunks (vs 207 in v1)

Retrieved 3 chunks:
   [0.7664] Meezan Bank | Meezan-Bank-FAQs-Digital-Account.pdf
   [0.7134] Meezan Bank | Meezan-Bank-FAQs-Roshan-Digital-Account.pdf
   [0.6817] Meezan Bank | Meezan-Bank-FAQs-Roshan-Digital-Account.pdf

Sending to Groq (llama-3.1-8b-instant)...

ANSWER:
   According to the context documents, no additional documents are required to open a Meezan Roshan Digital Account. The documents submitted at the time of account opening will suffice for Al Meezan's investment account opening process.

SOURCES USED:
   - Meezan-Bank-FAQs-Digital-Account.pdf
   - Meezan-Bank-FAQs-Ro

---
## Evaluation — Hit Rate + MRR

We evaluate **retrieval quality** on the 7 in-scope questions using the same two metrics as v1:

| Metric | What it measures |
|---|---|
| **Hit Rate** | Did the correct source appear anywhere in the top-K retrieved chunks? |
| **MRR** (Mean Reciprocal Rank) | How high was the correct source ranked? Rewards rank 1 more than rank 3. |

MRR formula: `MRR = mean(1 / rank_of_first_hit)` — a perfect score is `1.0` (always rank 1).

Results here are then compared side-by-side against v1 in Step 11.

In [15]:
# Only evaluate in-scope questions (those with an expected source)
in_scope_results = [r for r in all_results if r["expected_source"] is not None]

hits             = 0
reciprocal_ranks = []

for r in in_scope_results:
    # Reuse the _retrieved chunks stored in the result dict
    # avoids a redundant embed+search call per question during evaluation
    retrieved_for_eval = r.get("_retrieved") or retrieve_chunks_v2(r["question"], top_k=TOP_K)[0]
    retrieved_sources  = [c["source_file"] for c in retrieved_for_eval]
    expected           = r["expected_source"]

    # Find rank of first hit
    hit_rank = None
    for rank, src in enumerate(retrieved_sources, 1):
        if expected.lower() in src.lower():
            hit_rank = rank
            break

    hit = hit_rank is not None
    rr  = 1 / hit_rank if hit else 0.0

    hits += int(hit)
    reciprocal_ranks.append(rr)

    # Attach metrics back to result for the comparison table
    r["hit"]      = hit
    r["hit_rank"] = hit_rank
    r["rr"]       = rr

    cluster_used = r["cluster_info"]["predicted_cluster_id"]
    status       = f"Hit @ rank {hit_rank}" if hit else "Miss"
    print(f"{status:22s} | Cluster {cluster_used} | {r['question'][:55]}")

hit_rate_v2 = hits / len(in_scope_results)
mrr_v2      = float(np.mean(reciprocal_ranks))

print(f"\n{'='*50}")
print(f"Hit Rate v2 (top-{TOP_K}) : {hit_rate_v2:.2%}  ({hits}/{len(in_scope_results)})")
print(f"MRR v2                : {mrr_v2:.4f}")
print(f"{'='*50}")

Hit @ rank 1           | Cluster 4 | What is the cash transaction limit for HBL home remitta
Hit @ rank 2           | Cluster 0 | What documents are needed to open a Meezan Roshan Digit
Miss                   | Cluster 2 | Is net metering included in Meezan Bank solar panel fin
Hit @ rank 1           | Cluster 4 | What is the late payment charge on Alfalah personal loa
Hit @ rank 1           | Cluster 3 | Can a foreign national register for ABL myABL digital b
Hit @ rank 1           | Cluster 2 | What is the withholding tax rate on Pakistan Investment
Miss                   | Cluster 1 | Can Islamic banks charge penalty for late payment?

Hit Rate v2 (top-3) : 71.43%  (5/7)
MRR v2                : 0.6429


---
## v1 vs v2 Comparison

We load the v1 results saved by `rag-pipeline.ipynb` and compare them
side-by-side against v2 on Hit Rate, MRR, and search space reduction.

**What to look for:**
- Hit Rate and MRR should be equal or better in v2 — if they are worse, the cluster labels
  in `clustering.ipynb` may need to be revised or `K_FINAL` adjusted
- Search space reduction shows how many fewer chunks v2 evaluated per question on average

In [16]:
# Load v1 results for comparison
v1_results_path = Path("../../data/test/rag_test_results.json")

v1_loaded = False
if v1_results_path.exists():
    with open(v1_results_path, "r", encoding="utf-8") as f:
        v1_records = json.load(f)

    # Recompute v1 Hit Rate and MRR from saved records
    v1_in_scope = [r for r in v1_records if r["expected_source"] is not None]
    v1_hits     = sum(1 for r in v1_in_scope if r.get("hit"))
    v1_rrs      = [r.get("rr", 0.0) for r in v1_in_scope]
    hit_rate_v1 = v1_hits / len(v1_in_scope) if v1_in_scope else 0.0
    mrr_v1      = float(np.mean(v1_rrs)) if v1_rrs else 0.0
    v1_loaded   = True
    print(f"v1 results loaded from {v1_results_path}")
else:
    hit_rate_v1 = None
    mrr_v1      = None
    print(f"v1 results not found at {v1_results_path}")
    print("Run rag_pipeline_groq.ipynb first to enable comparison.")

# Average search space used by v2
avg_search_space = np.mean([
    r["cluster_info"]["cluster_size"] for r in all_results
])
reduction_pct = (1 - avg_search_space / len(chunks)) * 100

print(f"\n{'='*55}")
print(f"{'Metric':<30} {'v1':>10} {'v2':>10}")
print(f"{'-'*55}")
print(f"{'Hit Rate (top-' + str(TOP_K) + ')':<30} "
      f"{(f'{hit_rate_v1:.2%}' if hit_rate_v1 is not None else 'N/A'):>10} "
      f"{hit_rate_v2:.2%}:>10")
print(f"{'MRR':<30} "
      f"{(f'{mrr_v1:.4f}' if mrr_v1 is not None else 'N/A'):>10} "
      f"{mrr_v2:.4f}:>10")
print(f"{'Chunks searched (avg)':<30} {len(chunks):>10} {avg_search_space:>10.1f}")
print(f"{'Search space reduction':<30} {'—':>10} {reduction_pct:>9.1f}%")
print(f"{'='*55}")

v1 results not found at ..\..\data\test\rag_test_results.json
Run rag_pipeline_groq.ipynb first to enable comparison.

Metric                                 v1         v2
-------------------------------------------------------
Hit Rate (top-3)                      N/A 71.43%:>10
MRR                                   N/A 0.6429:>10
Chunks searched (avg)                 207       33.5
Search space reduction                  —      83.8%


---
## Results Summary Table + Save to JSON

In [17]:
# Build summary dataframe for all questions (including out-of-scope)
rows = []
for r in all_results:
    is_out_of_scope = r["expected_source"] is None

    if is_out_of_scope:
        retrieval_col = "N/A (out-of-scope)"
    else:
        retrieval_col = f"Hit @ rank {r.get('hit_rank')}" if r.get("hit") else "Miss"

    rows.append({
        "Question"        : r["question"][:50] + "...",
        "RAG Status"      : "Answered" if r["status"] == "answered" else "No Info",
        "Best Sim"        : r["best_similarity"],
        "Cluster Used"    : f"{r['cluster_info']['predicted_cluster_id']} — {r['cluster_info']['predicted_cluster_label'][:20]}",
        "Search Space"    : f"{r['cluster_info']['cluster_size']} / {len(chunks)}",
        "Retrieval"       : retrieval_col,
        "Answer Preview"  : r["answer"][:70] + "..."
    })

df_results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 260)

print("RAG PIPELINE OP — COMPLETE TEST RESULTS")
print("=" * 120)
print(df_results.to_string(index=False))

RAG PIPELINE OP — COMPLETE TEST RESULTS
                                             Question RAG Status  Best Sim             Cluster Used Search Space          Retrieval                                                            Answer Preview
What is the cash transaction limit for HBL home re...   Answered    0.7264 4 — International Transf     26 / 207       Hit @ rank 1 The cash transaction limit for HBL home remittance service is PKR 500,...
What documents are needed to open a Meezan Roshan ...   Answered    0.7664     0 — Digital Accounts     39 / 207       Hit @ rank 2 According to the context documents, no additional documents are requir...
Is net metering included in Meezan Bank solar pane...   Answered    0.7864 2 — Loans & Financing Pr     36 / 207               Miss             No, net metering is not part of financing (Source 1: Q29)....
What is the late payment charge on Alfalah persona...   Answered    0.5920 4 — International Transf     26 / 207       Hit @ rank 1 I do

In [18]:
# Save v2 results to JSON
save_records = []
for r in all_results:
    save_records.append({
        "question":           r["question"],
        "answer":             r["answer"],
        "sources":            r["sources"],
        "best_similarity":    r["best_similarity"],
        "status":             r["status"],
        "hit":                r.get("hit"),
        "hit_rank":           r.get("hit_rank"),
        "rr":                 r.get("rr"),
        "expected_source":    r.get("expected_source"),
        "predicted_cluster":  r["cluster_info"]["predicted_cluster_id"],
        "cluster_label":      r["cluster_info"]["predicted_cluster_label"],
        "cluster_size":       r["cluster_info"]["cluster_size"],
        "fallback_used":      r["cluster_info"]["fallback"],
    })

output_path = RAG_DIR / "rag_test_results_op.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(save_records, f, ensure_ascii=False, indent=2)

print(f"Results saved to : {output_path}")
print(f"   Total questions  : {len(save_records)}")
print(f"       Answered       : {sum(1 for r in save_records if r['status'] == 'answered')}")
print(f"       Below threshold: {sum(1 for r in save_records if r['status'] == 'below_threshold')}")
print(f"       Fallback used  : {sum(1 for r in save_records if r['fallback_used'])}")

Results saved to : ..\..\data\rag\rag_test_results_op.json
   Total questions  : 8
       Answered       : 7
       Below threshold: 1
       Fallback used  : 0


---
## Pipeline Summary

| Component | v1 Choice | v2 Choice | Change |
|---|---|---|---|
| Embedding Model | `all-MiniLM-L6-v2` | `all-MiniLM-L6-v2` | Same |
| Vector Store | FAISS `IndexFlatIP` | FAISS `IndexFlatIP` | Same |
| LLM | Llama 3.1 8B (Groq) | Llama 3.1 8B (Groq) | Same |
| Retrieval Scope | All 207 chunks | Predicted cluster only | Reduced search space |
| Cluster Prediction | None | Centroid dot product | New in v2 |
| Fallback | N/A | Full index if cluster too small | New in v2 |
| Prompt Strategy | Context-only grounding | Context-only grounding | Same |
| Similarity Threshold | 0.3 | 0.3 | Same |
| Top-K | 3 | 3 | Same |

---

### Files Generated After Running This Notebook

```
data/
├── embeddings/
|   ├── faiss_index.bin                    <- from embedding.ipynb (unchanged)
|   └── chunks_with_metadata.json          <- from embedding.ipynb (unchanged)
|
├── clustering/
|   └── chunks_with_clusters.json          <- from clustering.ipynb (unchanged)
|
└── test/
    ├── rag_test_results.json              <- from rag-pipeline.ipynb (v1)
    └── rag_test_results_v2.json           <- NEW: rag-pipeline_op.ipynb (v2)
```